In [1]:
# 1 — импорты и константы
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

RANDOM_STATE = 42
TARGET_COL = "flag"


In [8]:
# 2 — загрузка готового датасета
df = pd.read_pickle("../data/processed/dataset_with_target.pkl")

print(df.shape)
print(df[TARGET_COL].value_counts(normalize=True))


(3000000, 168)
flag
0    0.964519
1    0.035481
Name: proportion, dtype: float64


In [24]:
# просмотр содержимого df
print("shape:", df.shape)
print("\nПервые 30 колонок:")
print(df.columns[:30].tolist())

print("\nenc_paym колонки:")
print([c for c in df.columns if c.startswith("enc_paym_")])

print("\nКолонки *_max:")
print([c for c in df.columns if c.endswith("_max")][:20])

print("\nКолонки *_mean:")
print([c for c in df.columns if c.endswith("_mean")][:20])

shape: (3000000, 168)

Первые 30 колонок:
['id', 'rn_mean_mean', 'rn_mean_max', 'rn_max_mean', 'rn_max_max', 'pre_since_opened_mean_mean', 'pre_since_opened_mean_max', 'pre_since_opened_max_mean', 'pre_since_opened_max_max', 'pre_since_confirmed_mean_mean', 'pre_since_confirmed_mean_max', 'pre_since_confirmed_max_mean', 'pre_since_confirmed_max_max', 'pre_pterm_mean_mean', 'pre_pterm_mean_max', 'pre_pterm_max_mean', 'pre_pterm_max_max', 'pre_fterm_mean_mean', 'pre_fterm_mean_max', 'pre_fterm_max_mean', 'pre_fterm_max_max', 'pre_till_pclose_mean_mean', 'pre_till_pclose_mean_max', 'pre_till_pclose_max_mean', 'pre_till_pclose_max_max', 'pre_till_fclose_mean_mean', 'pre_till_fclose_mean_max', 'pre_till_fclose_max_mean', 'pre_till_fclose_max_max', 'pre_loans_credit_limit_mean_mean']

enc_paym колонки:
[]

Колонки *_max:
['rn_mean_max', 'rn_max_max', 'pre_since_opened_mean_max', 'pre_since_opened_max_max', 'pre_since_confirmed_mean_max', 'pre_since_confirmed_max_max', 'pre_pterm_mean_max', '

In [9]:
# ЯЧЕЙКА 2.1
# загрузка RAW parquet для извлечения enc_paym_*

df_raw = pd.read_parquet("../data/raw/train_data_0.pq")

print(df_raw.shape)
print(
    len([c for c in df_raw.columns if c.startswith("enc_paym_")]),
    "enc_paym_* колонок найдено"
)


(1974724, 61)
25 enc_paym_* колонок найдено


In [10]:
# 3 — подвыборка 300k
df_small, _ = train_test_split(
    df,
    train_size=300_000,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE
)

df_small = df_small.reset_index(drop=True)

print(df_small.shape)
print(df_small[TARGET_COL].value_counts(normalize=True))


(300000, 168)
flag
0    0.96452
1    0.03548
Name: proportion, dtype: float64


In [11]:
# 4 — baseline без дополнительных FE
X = df_small.drop(columns=[TARGET_COL])
y = df_small[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

baseline_auc = roc_auc_score(y_test, y_pred)
baseline_auc


0.7188457103442984

In [18]:
# ЯЧЕЙКА 5
# paym_bad_cnt как в эксперименте: считаем по max-статусу платежей на строке,
# затем суммируем по id (кол-во "плохих кредитных записей" / месяцев с плохим статусом)

paym_cols = [c for c in df_raw.columns if c.startswith("enc_paym_")]
assert len(paym_cols) > 0, "enc_paym_* колонки не найдены в df_raw"

tmp = df_raw[["id"] + paym_cols].copy()

# max-статус по всем enc_paym_* в строке
tmp["paym_max"] = tmp[paym_cols].max(axis=1)

# плохая запись = max >= 3
tmp["paym_bad_row"] = (tmp["paym_max"] >= 3).astype(int)

# итоговый счётчик по id
df_paym = (
    tmp[["id", "paym_bad_row"]]
    .groupby("id", as_index=False)
    .sum()
    .rename(columns={"paym_bad_row": "paym_bad_cnt"})
)

df_paym.head(), df_paym["paym_bad_cnt"].describe()




(   id  paym_bad_cnt
 0   0            10
 1   1            12
 2   2             3
 3   3            10
 4   4             1,
 count    250000.000000
 mean          6.820304
 std           5.113356
 min           0.000000
 25%           3.000000
 50%           6.000000
 75%           9.000000
 max          51.000000
 Name: paym_bad_cnt, dtype: float64)

In [19]:
# ЯЧЕЙКА 5.1
# мёрджим paym_bad_cnt в основной датасет

df_full = df.merge(df_paym, on="id", how="left")

df_full["paym_bad_cnt"] = (
    df_full["paym_bad_cnt"]
    .fillna(0)
    .astype(int)
)

df_full[["id", "paym_bad_cnt"]].head()


,id,paym_bad_cnt
0,0,10
1,1,12
2,2,3
3,3,10
4,4,1


In [20]:
# ЯЧЕЙКА 6
# interaction util × paym_bad_cnt

df_full = df_full.copy()

df_full["util_mean_x_paym_bad"] = (
    df_full["pre_util_mean_mean"] * df_full["paym_bad_cnt"]
)



In [21]:
# ЯЧЕЙКА 7
# подвыборка, как в эксперименте с 0.74

df_small, _ = train_test_split(
    df_full,
    train_size=300_000,
    stratify=df_full[TARGET_COL],
    random_state=RANDOM_STATE
)

df_small = df_small.reset_index(drop=True)

df_small.shape, df_small[TARGET_COL].value_counts(normalize=True)



((300000, 170),
 flag
 0    0.96452
 1    0.03548
 Name: proportion, dtype: float64)

In [22]:
# ЯЧЕЙКА 8
# baseline без paym_bad_cnt

X = df_small.drop(columns=[TARGET_COL, "paym_bad_cnt", "util_mean_x_paym_bad"])
y = df_small[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

baseline_auc = roc_auc_score(y_test, y_pred)
baseline_auc


0.7188457103442984

In [23]:
# ЯЧЕЙКА 9
# модель с count + interaction

X = df_small.drop(columns=[TARGET_COL])
y = df_small[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred)
auc, auc - baseline_auc


(0.7210057477167415, 0.00216003737244308)